In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)   # empty slices where a whole row was dropped
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

DATA = '../data'
X_train = np.load(f'{DATA}/train_cadences.npy', mmap_mode='r')   # (18000, 6, 16, 128) uint8
X_test  = np.load(f'{DATA}/test_cadences.npy', mmap_mode='r')    # (10000, 6, 16, 128) uint8
train = pd.read_csv(f'{DATA}/train_meta.csv')
test  = pd.read_csv(f'{DATA}/test_meta.csv')
print(X_train.shape, X_test.shape)
print(train.label.value_counts().to_dict())

(18000, 6, 16, 128) (10000, 6, 16, 128)
{'noise': 4211, 'rfi_persistent': 2892, 'rfi_intermittent': 2662, 'radar_chirp': 1849, 'satellite_pass': 1805, 'pulsar': 1629, 'scintillating_star': 1431, 'frb': 1042, 'eti_candidate': 479}


In [2]:
from pathlib import Path


FEATURE_DIR = Path("../outputs/features")

F_e3_train = pd.read_pickle(FEATURE_DIR / "F_e3_train.pkl")
F_e3_test = pd.read_pickle(FEATURE_DIR / "F_e3_test.pkl")

print("E3 train:", F_e3_train.shape)
print("E3 test: ", F_e3_test.shape)
print("Raw train:", X_train.shape)
print("Raw test: ", X_test.shape)

E3 train: (18000, 106)
E3 test:  (10000, 106)
Raw train: (18000, 6, 16, 128)
Raw test:  (10000, 6, 16, 128)


In [3]:
def curved_drift_features(
    X,
    batch_size=64,
    starts=None,
    slopes=None,
    curvatures=None,
):
    """
    Extract curved-drift features from X with shape
    (n_samples, 6, 16, 128).

    Zero-valued pixels are treated as missing.

    Returns five features per panel:
    score, start, slope, curvature, curved_gain.
    """

    X = np.asarray(X)
    n_samples, n_panels, n_times, n_freqs = X.shape

    if (n_panels, n_times, n_freqs) != (6, 16, 128):
        raise ValueError(f"Unexpected input shape: {X.shape}")

    # Candidate tracks are centered at the middle of the panel.
    starts = (
        np.arange(0, n_freqs, 2, dtype=float)
        if starts is None else np.asarray(starts, dtype=float)
    )
    slopes = (
        np.arange(-2.0, 2.01, 0.5)
        if slopes is None else np.asarray(slopes, dtype=float)
    )
    curvatures = (
        np.array([-0.04, 0.0, 0.04])
        if curvatures is None else np.asarray(curvatures, dtype=float)
    )

    times = np.arange(n_times, dtype=float) - (n_times - 1) / 2

    # Each candidate is described by its center frequency,
    # linear drift rate, and curvature.
    candidates = []
    for start in starts:
        for slope in slopes:
            for curvature in curvatures:
                frequency = (
                    start
                    + slope * times
                    + curvature * times**2
                )

                in_bounds = (
                    (frequency >= 0)
                    & (frequency < n_freqs)
                )

                indices = np.rint(frequency).astype(int)
                indices = np.clip(indices, 0, n_freqs - 1)

                candidates.append(
                    (start, slope, curvature, indices, in_bounds)
                )

    candidate_indices = np.stack([c[3] for c in candidates])
    candidate_bounds = np.stack([c[4] for c in candidates])

    candidate_starts = np.array([c[0] for c in candidates])
    candidate_slopes = np.array([c[1] for c in candidates])
    candidate_curvatures = np.array([c[2] for c in candidates])

    n_candidates = len(candidates)
    output = np.zeros((n_samples, n_panels * 5), dtype=np.float32)

    for panel_idx in range(n_panels):
        panel_features = np.zeros((n_samples, 5), dtype=np.float32)

        for batch_start in range(0, n_samples, batch_size):
            batch_end = min(batch_start + batch_size, n_samples)

            raw = X[batch_start:batch_end, panel_idx].astype(
                np.float32, copy=False
            )

            valid = raw != 0
            masked = np.where(valid, raw, np.nan)

            # Estimate the frequency-dependent background from time.
            with np.errstate(invalid="ignore"):
                channel_bg = np.nanmedian(
                    masked, axis=1, keepdims=True
                )

            residual = raw - channel_bg
            residual[~valid] = 0
            residual[~np.isfinite(residual)] = 0

            # Robust per-sample scale.
            abs_residual = np.where(valid, np.abs(residual), np.nan)
            with np.errstate(invalid="ignore"):
                scale = np.nanmedian(
                    abs_residual, axis=(1, 2), keepdims=True
                )

            scale = np.where(
                np.isfinite(scale) & (scale > 1e-6), scale, 1.0
            )
            residual = residual / scale

            # Gather pixels along every candidate track.
            gathered = np.take_along_axis(
                residual[:, None, :, :],
                candidate_indices[None, :, :, None],
                axis=3,
            )[..., 0]

            gathered_valid = np.take_along_axis(
                valid[:, None, :, :],
                candidate_indices[None, :, :, None],
                axis=3,
            )[..., 0]

            track_valid = (
                gathered_valid
                & candidate_bounds[None, :, :]
            )

            counts = track_valid.sum(axis=2)

            # Score positive excess along a track, accounting for
            # the number of observed pixels.
            positive_signal = np.where(
                track_valid, np.maximum(gathered, 0), 0
            )
            scores = positive_signal.sum(axis=2) / np.sqrt(
                np.maximum(counts, 1)
            )

            # Avoid trusting tracks supported by too few pixels.
            scores[counts < 8] = -np.inf

            # Samples with no valid candidate get neutral features.
            has_candidate = np.isfinite(scores).any(axis=1)
            safe_scores = np.where(np.isfinite(scores), scores, -np.inf)

            best_idx = np.argmax(safe_scores, axis=1)
            rows = np.arange(batch_end - batch_start)

            best_score = safe_scores[rows, best_idx]
            best_score = np.where(has_candidate, best_score, 0.0)

            best_start = candidate_starts[best_idx]
            best_slope = candidate_slopes[best_idx]
            best_curvature = candidate_curvatures[best_idx]
            best_coverage = counts[rows, best_idx] / n_times

            # Compare the best candidate against the best straight track.
            straight_mask = candidate_curvatures == 0
            straight_scores = safe_scores[:, straight_mask]
            best_straight = np.max(straight_scores, axis=1)
            best_straight = np.where(
                np.isfinite(best_straight), best_straight, 0.0
            )

            curved_gain = np.maximum(
                best_score - best_straight, 0.0
            )

            panel_features[batch_start:batch_end] = np.column_stack([
                best_score,
                best_start,
                best_slope,
                best_curvature,
                curved_gain,
            ])

        output[:, panel_idx * 5:(panel_idx + 1) * 5] = panel_features

    columns = [
        f"curve_p{p}_{feature}"
        for p in range(6)
        for feature in ("score", "start", "slope", "curvature", "curved_gain")
    ]

    return pd.DataFrame(output, columns=columns)

In [4]:
D4_train = curved_drift_features(X_train)
D4_test = curved_drift_features(X_test)

print("Curved features, train:", D4_train.shape)
print("Curved features, test: ", D4_test.shape)

D4_train.to_pickle(FEATURE_DIR / "D4_train.pkl")
D4_test.to_pickle(FEATURE_DIR / "D4_test.pkl")

Curved features, train: (18000, 30)
Curved features, test:  (10000, 30)


In [5]:
display(D4_train.describe().T)

,count,mean,std,min,25%,50%,75%,max
curve_p0_score,18000.0,6.387522,2.478687,3.189655,5.386812,5.741254,6.265886,36.666664
curve_p0_start,18000.0,63.149113,35.080276,0.000000,32.000000,62.000000,94.000000,126.000000
curve_p0_slope,18000.0,-0.009222,1.236539,-2.000000,-1.000000,0.000000,1.000000,2.000000
curve_p0_curvature,18000.0,-0.000358,0.032172,-0.040000,-0.040000,0.000000,0.040000,0.040000
curve_p0_curved_gain,18000.0,0.388439,0.515675,0.000000,0.000000,0.218750,0.609375,6.765625
curve_p1_score,18000.0,6.262260,2.209281,4.323529,5.421875,5.750000,6.183333,35.109375
curve_p1_start,18000.0,63.406113,35.040634,0.000000,34.000000,64.000000,94.000000,126.000000
curve_p1_slope,18000.0,-0.004556,1.258012,-2.000000,-1.000000,0.000000,1.000000,2.000000
curve_p1_curvature,18000.0,-0.000549,0.032568,-0.040000,-0.040000,0.000000,0.040000,0.040000
curve_p1_curved_gain,18000.0,0.387190,0.476321,0.000000,0.000000,0.236111,0.617647,5.031250


In [6]:
print("Curvature value counts:")
display(D4_train.filter(like="_curvature").stack().value_counts().head(10))

print("\nCurved gain summary:")
display(D4_train.filter(like="_curved_gain").describe())

Curvature value counts:


 0.00    36622
-0.04    35814
 0.04    35564
Name: count, dtype: int64


Curved gain summary:


,curve_p0_curved_gain,curve_p1_curved_gain,curve_p2_curved_gain,curve_p3_curved_gain,curve_p4_curved_gain,curve_p5_curved_gain
count,18000.000000,18000.000000,18000.000000,18000.000000,18000.000000,18000.000000
mean,0.388439,0.387190,0.396387,0.386272,0.391204,0.383814
std,0.515675,0.476321,0.518404,0.480615,0.510280,0.472318
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.218750,0.236111,0.234375,0.233334,0.233333,0.250000
75%,0.609375,0.617647,0.613275,0.616667,0.609375,0.612147
max,6.765625,5.031250,7.593750,6.150002,5.899922,6.966667


In [9]:
F_e4_train = pd.concat(
    [
        F_e3_train.reset_index(drop=True),
        D4_train.reset_index(drop=True),
    ],
    axis=1,
)

F_e4_test = pd.concat(
    [
        F_e3_test.reset_index(drop=True),
        D4_test.reset_index(drop=True),
    ],
    axis=1,
)

# Validate row counts and train/test feature consistency.
assert len(F_e4_train) == len(F_e3_train) == len(D4_train)
assert len(F_e4_test) == len(F_e3_test) == len(D4_test)
assert F_e4_train.shape[1] == F_e3_train.shape[1] + D4_train.shape[1]
assert F_e4_test.shape[1] == F_e3_test.shape[1] + D4_test.shape[1]
assert F_e4_train.columns.equals(F_e4_test.columns)

F_e4_train.to_pickle(FEATURE_DIR / "F_e4_train.pkl")
F_e4_test.to_pickle(FEATURE_DIR / "F_e4_test.pkl")

print("E4 train:", F_e4_train.shape)
print("E4 test: ", F_e4_test.shape)

E4 train: (18000, 136)
E4 test:  (10000, 136)


In [10]:

from sklearn.metrics import f1_score, classification_report

y = train["label"].reset_index(drop=True)

X3_tr, X3_val, y_tr, y_val = train_test_split(
    F_e3_train,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

X4_tr, X4_val, y4_tr, y4_val = train_test_split(
    F_e4_train,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

assert y_tr.equals(y4_tr)
assert y_val.equals(y4_val)

results = []

for learning_rate in [0.02, 0.03, 0.04, 0.05, 0.06]:
    model = HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=learning_rate,
        class_weight="balanced",
        random_state=0,
    )

    model.fit(X4_tr, y4_tr)
    val_pred = model.predict(X4_val)
    score = f1_score(y_val, val_pred, average="macro")

    results.append({
        "learning_rate": learning_rate,
        "macro_f1": score,
    })

results_df = pd.DataFrame(results).sort_values(
    "macro_f1", ascending=False
)
display(results_df)

,learning_rate,macro_f1
4,0.06,0.608898
1,0.03,0.608568
2,0.04,0.607897
3,0.05,0.603689
0,0.02,0.603401


In [11]:
e3_results = []

for learning_rate in [0.02, 0.03, 0.04, 0.05, 0.06]:
    model = HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=learning_rate,
        class_weight="balanced",
        random_state=0,
    )

    model.fit(X3_tr, y_tr)
    val_pred = model.predict(X3_val)

    e3_results.append({
        "learning_rate": learning_rate,
        "macro_f1": f1_score(y_val, val_pred, average="macro"),
    })

e3_results_df = pd.DataFrame(e3_results)

comparison = (
    e3_results_df.rename(columns={"macro_f1": "E3_macro_f1"})
    .merge(
        results_df.rename(columns={"macro_f1": "E4_macro_f1"}),
        on="learning_rate",
    )
)

comparison["gain"] = (
    comparison["E4_macro_f1"] - comparison["E3_macro_f1"]
)

display(comparison.sort_values("learning_rate"))

,learning_rate,E3_macro_f1,E4_macro_f1,gain
0,0.02,0.609301,0.603401,-0.005900
1,0.03,0.608830,0.608568,-0.000262
2,0.04,0.612550,0.607897,-0.004653
3,0.05,0.614095,0.603689,-0.010406
4,0.06,0.599587,0.608898,0.009311


In [12]:
from sklearn.metrics import classification_report, f1_score
from sklearn.ensemble import HistGradientBoostingClassifier

# E3 at learning rate 0.06
model_e3_06 = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.06,
    class_weight="balanced",
    random_state=0,
)
model_e3_06.fit(X3_tr, y_tr)
pred_e3_06 = model_e3_06.predict(X3_val)

# E4 at learning rate 0.06
model_e4_06 = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.06,
    class_weight="balanced",
    random_state=0,
)
model_e4_06.fit(X4_tr, y_tr)
pred_e4_06 = model_e4_06.predict(X4_val)

report_e3 = classification_report(
    y_val, pred_e3_06, output_dict=True, zero_division=0
)
report_e4 = classification_report(
    y_val, pred_e4_06, output_dict=True, zero_division=0
)

class_comparison = pd.DataFrame({
    "E3_F1": {
        label: report_e3[label]["f1-score"]
        for label in model_e3_06.classes_
    },
    "E4_F1": {
        label: report_e4[label]["f1-score"]
        for label in model_e4_06.classes_
    },
})

class_comparison["gain"] = (
    class_comparison["E4_F1"] - class_comparison["E3_F1"]
)

display(class_comparison.sort_values("gain"))

,E3_F1,E4_F1,gain
noise,0.799572,0.793415,-0.006158
rfi_persistent,0.734217,0.731066,-0.003150
scintillating_star,0.230366,0.230088,-0.000278
radar_chirp,0.735414,0.736264,0.000850
rfi_intermittent,0.589912,0.601751,0.011838
frb,0.698337,0.711538,0.013201
satellite_pass,0.540707,0.554364,0.013658
pulsar,0.848780,0.867863,0.019082
eti_candidate,0.218978,0.253731,0.034753


In [13]:


y = train["label"].reset_index(drop=True)

# Use the exact same validation split.
indices = pd.RangeIndex(len(y))
idx_tr, idx_val = train_test_split(
    indices,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

# Remove only the curved_gain columns.
D4_no_gain_train = D4_train.drop(
    columns=[c for c in D4_train.columns if c.endswith("_curved_gain")]
)
D4_no_gain_test = D4_test.drop(
    columns=[c for c in D4_test.columns if c.endswith("_curved_gain")]
)

F4_no_gain = pd.concat(
    [
        F_e3_train.reset_index(drop=True),
        D4_no_gain_train.reset_index(drop=True),
    ],
    axis=1,
)

experiments = {
    "E3": F_e3_train,
    "E4_full": F_e4_train,
    "E4_no_gain": F4_no_gain,
}

ablation_results = []

for name, features in experiments.items():
    for lr in [0.05, 0.06]:
        model = HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=lr,
            class_weight="balanced",
            random_state=0,
        )

        model.fit(features.iloc[idx_tr], y.iloc[idx_tr])
        pred = model.predict(features.iloc[idx_val])

        ablation_results.append({
            "experiment": name,
            "learning_rate": lr,
            "macro_f1": f1_score(
                y.iloc[idx_val], pred, average="macro"
            ),
        })

ablation_df = pd.DataFrame(ablation_results)
display(ablation_df.sort_values(["learning_rate", "macro_f1"],
                                ascending=[True, False]))

,experiment,learning_rate,macro_f1
0,E3,0.05,0.614095
4,E4_no_gain,0.05,0.604230
2,E4_full,0.05,0.603689
3,E4_full,0.06,0.608898
5,E4_no_gain,0.06,0.603037
1,E3,0.06,0.599587


In [14]:
from pathlib import Path
import json

RESULT_DIR = Path("../outputs")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

e4_summary = {
    "experiment": "E4_curved_drift",
    "baseline": "E3_on_off_coherence",
    "e3_best_macro_f1": 0.614095,
    "e3_best_learning_rate": 0.05,
    "e4_full_lr_005_macro_f1": 0.603689,
    "e4_full_lr_006_macro_f1": 0.608898,
    "e4_no_gain_lr_005_macro_f1": 0.604230,
    "e4_no_gain_lr_006_macro_f1": 0.603037,
    "decision": "Do not promote E4; retain E3 as validation baseline",
}

with open(RESULT_DIR / "experiment_4_summary.json", "w") as f:
    json.dump(e4_summary, f, indent=2)

print("E4 experiment summary saved.")

E4 experiment summary saved.
